# 03 - Classify support intents with word embeddings

Inputs: `amazon_help_labeled.csv` (from notebook 02). Builds domain **Word2Vec** embeddings, **visualizes the NMF intents** in embedding space, and trains a **logistic-regression intent classifier**.

On Kaggle: attach the dataset that 02 produced as a `gbm`-style output, or run immediately after 02 so its output lands in `/kaggle/working`. Locally it falls back to `../data/processed/amazon_help_labeled.csv`.

In [1]:
# install dependent packages (gensim is preinstalled on Kaggle)
!pip install -q gensim joblib


In [2]:
import os, re, json, time, warnings
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from gensim.models import Word2Vec
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from scipy.sparse import hstack
from concurrent.futures import ThreadPoolExecutor
import joblib

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110

SEED = 42
VEC_SIZE = 100
MIN_COUNT = 2
t0 = time.time()


In [9]:
# resolve input: Kaggle dataset (attach 'amazon-help-labeled' from 02 output), /working, or local fallback
LABELS_PATH = '/kaggle/input/amazon-help-labeled/amazon_help_labeled.csv'
LOCAL_PATH = '../data/processed/amazon_help_labeled.csv'

if os.path.exists(LABELS_PATH):
    DATA_PATH = LABELS_PATH
elif os.path.exists('/kaggle/working/amazon_help_labeled.csv'):
    DATA_PATH = '/kaggle/working/amazon_help_labeled.csv'
elif os.path.exists(LOCAL_PATH):
    DATA_PATH = LOCAL_PATH
else:
    DATA_PATH = LOCAL_PATH

df = pd.read_csv(DATA_PATH)
print(f'Loaded: {len(df)} rows from {DATA_PATH}')
df.head()


Loaded: 31972 rows from /kaggle/input/datasets/ayushdubey123456/amazon-help-labelled/amazon_help_labeled.csv


,tweet_id,author_id,text,created_at,topic_id,topic_confidence,intent,low_confidence
0,1000118,356779,it be all good now smilingfacewithsmilingeyes ...,Sun Oct 22 17:45:55 +0000 2017,0,0.016380,delivery_issue,False
1,1000129,356783,for a while every time i try to sign up for it...,Sun Oct 22 21:26:29 +0000 2017,4,0.054607,appreciation,False
2,1000132,356784,order id people need to know about duplicate p...,Sun Oct 22 19:12:01 +0000 2017,2,0.037112,order_status,False
3,1000136,356784,i know it very well,Sun Oct 22 19:43:47 +0000 2017,1,0.015487,customer_service,True
4,1000138,356784,i already connect do live chat you be ready fo...,Sun Oct 22 19:26:08 +0000 2017,1,0.023656,customer_service,False


## 1. Text to tokens
Strip stopwords and keep alpha tokens, mirroring the cleaned corpus style.

In [10]:
STOP = set(ENGLISH_STOP_WORDS)

def tokenize(text):
    tokens = re.findall(r"[a-z]+", str(text).lower())
    return [t for t in tokens if len(t) > 1 and t not in STOP]

print('Tokenizing...')
with ThreadPoolExecutor(max_workers=8) as ex:
    df['tokens'] = list(ex.map(tokenize, df['text']))

df['n_tokens'] = df['tokens'].str.len()
df = df[df['n_tokens'] > 0].reset_index(drop=True)

print(df.groupby("intent").size())
print(f'W2V corpus: {df["tokens"].str.len().sum():,} tokens')


Tokenizing...
intent
appreciation         3612
customer_service    12675
delivery_issue       6592
email_contact        4511
order_status         4492
dtype: int64
W2V corpus: 263,957 tokens


## 2. Train Word2Vec word embeddings
Learn distributional word vectors directly from the domain corpus (CBOW, 100-dim).

In [11]:
print("Training Word2Vec (CBOW, vector_size=100)...")
w2v = Word2Vec(sentences=df['tokens'], vector_size=VEC_SIZE, window=5,
               min_count=MIN_COUNT, workers=4, epochs=5, seed=SEED)
print(f"Vocab size: {len(w2v.wv)}")


Training Word2Vec (CBOW, vector_size=100)...
Vocab size: 6594


## 3. Embed tweets
A tweet = mean of its word vectors, then L2-normalized so cosine distance makes sense.

In [12]:
def embed_tweet(tokens):
    vectors = [w2v.wv[t] for t in tokens if t in w2v.wv]
    return np.mean(vectors, axis=0) if vectors else None

print("Embedding tweets...")
with ThreadPoolExecutor(max_workers=8) as ex:
    vecs = list(ex.map(embed_tweet, df['tokens']))

df['embedding'] = vecs
oov = int(df["embedding"].isna().sum())
df = df[df["embedding"].notna()].reset_index(drop=True)
print(f"Embedded: {len(df)} rows (dropped {oov} fully-OOV)")

emb = np.array(df["embedding"].tolist(), dtype=np.float32)
emb = normalize(emb, norm="l2")
print(f"Embedding matrix: {emb.shape}")

# save embeddings for reuse
OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '../data/processed'
np.savez_compressed(f"{OUT_DIR}/embeddings.npz",
                    tweet_id=df["tweet_id"].values, embedding=emb,
                    intent=df["intent"].values,
                    topic_confidence=df["topic_confidence"].values)
print(f"Saved: {OUT_DIR}/embeddings.npz")


Embedding tweets...
Embedded: 31862 rows (dropped 20 fully-OOV)
Embedding matrix: (31862, 100)
Saved: /kaggle/working/embeddings.npz


In [14]:
# ensure intent_mapping.json is available (from attached 'amazon-help-labeled' dataset if absent)
if not os.path.exists(f"{OUT_DIR}/intent_mapping.json"):
    import shutil
    for cand in ['/kaggle/input/amazon-help-labeled/intent_mapping.json',
                 '/kaggle/input/amazon-help-labeled/amazon_help_labeled/intent_mapping.json']:
        if os.path.exists(cand):
            shutil.copy(cand, f"{OUT_DIR}/intent_mapping.json")
            print(f"Copied intent_mapping.json from {cand}")
            break
assert os.path.exists(f"{OUT_DIR}/intent_mapping.json"), (
    'intent_mapping.json not found - run notebook 02 first or attach its output dataset.')


Copied intent_mapping.json from /kaggle/input/datasets/ayushdubey123456/amazon-help-labelled/intent_mapping.json


## 4. See the intents in embedding space

*Do the NMF intent clusters look separable in the embedding space?* Two views:
- **Word level** (left): PCA of the top-8 words per intent.
- **Tweet level** (right): t-SNE of a stratified subsample of tweet embeddings, colored by intent.

In [15]:
with open(f"{OUT_DIR}/intent_mapping.json") as f:
    mapping = json.load(f)

labels_ordered = list(df["intent"].unique())
intent_order = [mapping["topic_labels"][str(t)] for t in sorted(map(int, mapping["topic_labels"]))]
label_to_topic = {v: int(k) for k, v in mapping["topic_labels"].items()}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
colors = plt.cm.tab10(np.linspace(0, 1, len(intent_order)))
color_map = dict(zip(intent_order, colors))

# Panel A - word embedding space
all_words, word_intents, w_vectors = [], [], []
for intent in intent_order:
    for w in mapping["top_words"][str(label_to_topic[intent])][:8]:
        if w in w2v.wv:
            all_words.append(w); word_intents.append(intent); w_vectors.append(w2v.wv[w])
w_pca = PCA(n_components=2, random_state=SEED).fit_transform(normalize(np.array(w_vectors), norm="l2"))
for i, (x, y) in enumerate(w_pca):
    axes[0].scatter(x, y, color=color_map[word_intents[i]], s=60,
                    edgecolors="black", linewidths=0.4)
    axes[0].annotate(all_words[i], (x, y), fontsize=8, alpha=0.85)
axes[0].set_title("Word embeddings (PCA) - top words per intent")

# Panel B - tweet embedding space (stratified subsample)
rng = np.random.RandomState(SEED)
sub_idx = np.concatenate([
    rng.choice(np.where(df["intent"].values == intent)[0],
               size=min(1200, int((df["intent"].values == intent).sum())), replace=False)
    for intent in intent_order])
sub_emb = emb[sub_idx]; sub_int = df["intent"].values[sub_idx]
tsne = TSNE(n_components=2, perplexity=30, init="pca", random_state=SEED, method="barnes_hut")
tsne_xy = tsne.fit_transform(sub_emb)
for intent in intent_order:
    mask = sub_int == intent
    axes[1].scatter(tsne_xy[mask, 0], tsne_xy[mask, 1], s=4, alpha=0.55,
                    color=color_map[intent], label=intent)
axes[1].set_title("Tweet embeddings (t-SNE, subsample)")
axes[1].legend(markerscale=6, fontsize=8)

handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=color_map[i],
                      markersize=8, label=i) for i in intent_order]
fig.suptitle("Embedding Space: Do the NMF intents look separable?", fontsize=13)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/embedding_visualization.png")
plt.show()
print(f"Saved: {OUT_DIR}/embedding_visualization.png")


Saved: /kaggle/working/embedding_visualization.png


## 5. Classify intent
Train a logistic-regression classifier on the tweet embeddings. Two important choices:
- **Silver labels** come from NMF topic assignment, so we train only on *high-confidence* rows (75%) and evaluate on a hold-out that mixes all labels.
- Stratified train / val / test split (70 / 15 / 15).

In [16]:
y = LabelEncoder().fit(df["intent"])
y_enc = y.transform(df["intent"])
X = emb

X_tr, X_te, y_tr, y_te, idx_tr, idx_te = train_test_split(
    X, y_enc, np.arange(len(df)), test_size=0.15, random_state=SEED, stratify=y_enc)
X_tr, X_val, y_tr, y_val, idx_tr, idx_val = train_test_split(
    X_tr, y_tr, idx_tr, test_size=0.1765, random_state=SEED, stratify=y_tr)

train_conf = ~df["low_confidence"].values[idx_tr]
sup_tr, y_sup = X_tr[train_conf], y_tr[train_conf]
print(f"train={len(X_tr)} (high-conf: {len(sup_tr)})  val={len(X_val)}  test={len(X_te)}")

clf = LogisticRegression(max_iter=2000, C=1.0, random_state=SEED)
clf.fit(sup_tr, y_sup)
val_acc = accuracy_score(y_val, clf.predict(X_val))
val_f1 = f1_score(y_val, clf.predict(X_val), average='macro')
print(f'Validation accuracy: {val_acc:.4f}  macro-F1: {val_f1:.4f}')


train=22302 (high-conf: 16839)  val=4780  test=4780
Validation accuracy: 0.7533  macro-F1: 0.7386


In [19]:
y_pred = clf.predict(X_te)
print("Test performance:")
print(f"  accuracy:  {accuracy_score(y_te, y_pred):.4f}")
print(f"  macro-F1:  {f1_score(y_te, y_pred, average='macro'):.4f}")
print(classification_report(y_te, y_pred, target_names=y.classes_, zero_division=0))

cm = confusion_matrix(y_te, y_pred)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(y.classes_)))
ax.set_xticklabels(y.classes_, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(y.classes_)))
ax.set_yticklabels(y.classes_, fontsize=8)
for i in range(len(y.classes_)):
    for j in range(len(y.classes_)):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion matrix - intent classifier (test set)")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/confusion_matrix.png")
plt.show()
print(f"Saved: {OUT_DIR}/confusion_matrix.png")


Test performance:
  accuracy:  0.7500
  macro-F1:  0.7305
                  precision    recall  f1-score   support

    appreciation       0.49      0.61      0.54       542
customer_service       0.80      0.79      0.79      1901
  delivery_issue       0.76      0.76      0.76       986
   email_contact       0.77      0.72      0.74       677
    order_status       0.86      0.78      0.82       674

        accuracy                           0.75      4780
       macro avg       0.73      0.73      0.73      4780
    weighted avg       0.76      0.75      0.75      4780

Saved: /kaggle/working/confusion_matrix.png


## 6. Compare features: word2vec vs TF-IDF vs combined
Same classifier, same labels - only the feature vector changes. Note TF-IDF largely *reproduces* the NMF assignment (NMF ran on TF-IDF), so the honest generalization number is the embedding score.

In [18]:
from scipy.sparse import hstack
tfidf = TfidfVectorizer(max_features=None, ngram_range=(1, 1), min_df=2, max_df=0.5,
                        lowercase=True, stop_words="english")
X_tr_tf = tfidf.fit_transform([str(t) for t in df["text"].values[idx_tr]])
X_val_tf = tfidf.transform([str(t) for t in df["text"].values[idx_val]])
X_te_tf = tfidf.transform([str(t) for t in df["text"].values[idx_te]])

def lr_acc(Xa, ya, Xb, yb):
    m = LogisticRegression(max_iter=2000, C=1.0, random_state=SEED)
    m.fit(Xa[train_conf], y_tr[train_conf])
    return accuracy_score(yb, m.predict(Xb))

compare = {
    "word2vec_mean": {"val_acc": accuracy_score(y_val, clf.predict(X_val)),
                      "test_acc": accuracy_score(y_te, y_pred)},
    "tfidf": {"val_acc": lr_acc(X_tr_tf, y_tr, X_val_tf, y_val),
              "test_acc": lr_acc(X_tr_tf, y_tr, X_te_tf, y_te)},
}
X_tr_mix = hstack([X_tr, X_tr_tf]).tocsr()
X_val_mix = hstack([X_val, X_val_tf]).tocsr()
X_te_mix = hstack([X_te, X_te_tf]).tocsr()
compare["w2v+tfidf"] = {"val_acc": lr_acc(X_tr_mix, y_tr, X_val_mix, y_val),
                        "test_acc": lr_acc(X_tr_mix, y_tr, X_te_mix, y_te)}

for name, s in compare.items():
    print(f"  {name:14s}  val_acc={s['val_acc']:.4f}  test_acc={s['test_acc']:.4f}")


  word2vec_mean   val_acc=0.7533  test_acc=0.7500
  tfidf           val_acc=0.9259  test_acc=0.9243
  w2v+tfidf       val_acc=0.9266  test_acc=0.9209


## 7. Predict every tweet and save artifacts
Assign the classifier intent to all rows, flag prediction confidence, and persist the model + report.

In [20]:
df["pred_intent"] = y.inverse_transform(clf.predict(X))
df["pred_confidence"] = np.max(clf.predict_proba(X), axis=1)

save_cols = ["tweet_id", "author_id", "created_at", "text",
             "intent", "low_confidence", "topic_confidence", "pred_intent", "pred_confidence"]
df[save_cols].to_csv(f"{OUT_DIR}/amazon_help_classified.csv", index=False)
print(f"Saved: {OUT_DIR}/amazon_help_classified.csv ({len(df)} rows)")

MODEL_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '../data/models'
os.makedirs(MODEL_DIR, exist_ok=True)
w2v.save(f"{MODEL_DIR}/word2vec.model")
joblib.dump(clf, f"{MODEL_DIR}/intent_classifier.joblib")
joblib.dump(y, f"{MODEL_DIR}/label_encoder.joblib")
print(f"Saved: {MODEL_DIR}/word2vec.model, intent_classifier.joblib, label_encoder.joblib")

# --- TF-IDF classifier (saved for the tfidf engine; mirrors the notebook training split) ---
tfidf_clf = LogisticRegression(max_iter=2000, C=1.0, random_state=SEED)
tfidf_clf.fit(X_tr_tf[train_conf], y_tr[train_conf])
tfidf_val_acc = accuracy_score(y_val, tfidf_clf.predict(X_val_tf))
tfidf_test_acc = accuracy_score(y_te, tfidf_clf.predict(X_te_tf))
tfidf_test_f1 = f1_score(y_te, tfidf_clf.predict(X_te_tf), average='macro')
print(f'TF-IDF: val_acc={tfidf_val_acc:.4f} test_acc={tfidf_test_acc:.4f} macro-F1={tfidf_test_f1:.4f}')

joblib.dump(tfidf, f"{MODEL_DIR}/tfidf_vectorizer.joblib")
joblib.dump(tfidf_clf, f"{MODEL_DIR}/tfidf_classifier.joblib")
joblib.dump(y, f"{MODEL_DIR}/label_encoder_tfidf.joblib")
print(f"Saved: {MODEL_DIR}/tfidf_vectorizer.joblib, tfidf_classifier.joblib, label_encoder_tfidf.joblib")

report = {
    "model": str(clf),
    "features": f"word2vec_mean (vector_size={VEC_SIZE}) L2-normalized",
    "training": "high-confidence silver labels only",
    "val_accuracy": float(accuracy_score(y_val, clf.predict(X_val))),
    "val_macro_f1": float(f1_score(y_val, clf.predict(X_val), average="macro")),
    "test_accuracy": float(accuracy_score(y_te, y_pred)),
    "test_macro_f1": float(f1_score(y_te, y_pred, average="macro")),
    "per_class": classification_report(y_te, y_pred, target_names=y.classes_,
                                      output_dict=True, zero_division=0),
    "feature_comparison": compare,
    "tfidf": {
        "model": str(tfidf_clf),
        "features": "TfidfVectorizer(min_df=2, max_df=0.5, stop_words=english, unigrams)",
        "training": "high-confidence silver labels only",
        "val_accuracy": float(tfidf_val_acc),
        "test_accuracy": float(tfidf_test_acc),
        "test_macro_f1": float(tfidf_test_f1),
    },
    "label_mapping": {str(i): c for i, c in enumerate(y.classes_)},
}
with open(f"{OUT_DIR}/classification_report.json", "w") as f:
    json.dump(report, f, indent=2)
print(f"Saved: {OUT_DIR}/classification_report.json")


Saved: /kaggle/working/amazon_help_classified.csv (31862 rows)
Saved: /kaggle/working/word2vec.model, intent_classifier.joblib, label_encoder.joblib
TF-IDF: val_acc=0.9259 test_acc=0.9243 macro-F1=0.9241
Saved: /kaggle/working/tfidf_vectorizer.joblib, tfidf_classifier.joblib, label_encoder_tfidf.joblib
Saved: /kaggle/working/classification_report.json


## Findings so far
- **Embeddings learn real intent structure**: word2vec-mean gives ~0.76 accuracy and 0.74 macro-F1 with logistic regression, and the t-SNE view shows the five NMF intents forming coherent (if overlapping) clusters rather than noise.
- **TF-IDF beats embeddings for replaying this task** (~0.92) because the *labels were generated from TF-IDF* in the first place; the embedding score is the more honest generalization number.
- Next levers: tuned skip-gram / subword info, larger dims, sentence-transformers, or reducing label noise by acting on the 25% low-confidence rows.